In [ ]:
!pip install scikit-posthocs -q
!pip install ucimlrepo
 
import os, math, random, time
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
from scipy import stats as scipy_stats
from scipy.stats import rankdata
from collections import Counter
import seaborn as sns
import warnings

In [ ]:

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU: {gpus[0].name}")
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
 
class EnsembleDeepRVFL_GPU:
    def __init__(self, n_nodes=100, lam=0.01, n_layer=10):
        self.n_nodes = n_nodes
        self.lam     = lam
        self.n_layer = n_layer
        self.betas   = []
        self.weights = []
        self.biases  = []
        self.train_mean = None
        self.train_std  = None
 
    def train(self, X_np, y_np, n_class):
        X = tf.constant(X_np, dtype=tf.float32)
        y = tf.one_hot(y_np, n_class, dtype=tf.float32)
        
        unique_classes, class_counts = np.unique(y_np, return_counts=True)
        # Balanced class weights formula: N_total / (N_classes * N_c)
        class_weights = len(y_np) / (n_class * class_counts)
        
        # Map each sample to its corresponding class weight
        sample_weights = np.zeros(len(y_np), dtype=np.float32)
        for i, c in enumerate(unique_classes):
            sample_weights[y_np == c] = class_weights[i]
        
        W = tf.constant(sample_weights, dtype=tf.float32)
        sqrt_W = tf.sqrt(W)
        sqrt_W = tf.expand_dims(sqrt_W, 1) # Shape: (N, 1) for broadcasting
        # ======================================================================
        
        self.train_mean = tf.reduce_mean(X, axis=0)
        self.train_std  = tf.math.reduce_std(X, axis=0) + 1e-8
        X_st = (X - self.train_mean) / self.train_std
        input_data = X_st
        h = X_st
        
        for i in range(self.n_layer):
            w_dim = h.shape[1]
            w = tf.random.uniform([w_dim, self.n_nodes], minval=-1, maxval=1)
            b = tf.random.uniform([1,  self.n_nodes], minval=0,  maxval=1)
            h    = tf.nn.relu(tf.matmul(h, w) + b)
            H_l  = tf.concat([input_data, h], axis=1)
            batch_size = tf.shape(H_l)[0]
            H_aug = tf.concat([H_l, tf.ones([batch_size, 1], dtype=tf.float32)], axis=1)
            
            # ==================================================================
            # APPLY WEIGHTS TO THE LEAST SQUARES FORMULATION (Weighted Ridge)
            # ==================================================================
            H_weighted = H_aug * sqrt_W
            y_weighted = y * sqrt_W
            
            HT_H  = tf.matmul(H_weighted, H_weighted, transpose_a=True)
            HT_Y  = tf.matmul(H_weighted, y_weighted, transpose_a=True)
            # ==================================================================
            
            reg   = self.lam + 1e-6
            A     = HT_H + reg * tf.eye(tf.shape(HT_H)[0], dtype=tf.float32)
            try:
                beta = tf.linalg.solve(A, HT_Y)
            except tf.errors.InvalidArgumentError:
                beta = tf.matmul(tf.linalg.pinv(A), HT_Y)
            self.betas.append(beta)
            self.weights.append(w)
            self.biases.append(b)
 
    def predict(self, X_np):
        X    = tf.constant(X_np, dtype=tf.float32)
        X_st = (X - self.train_mean) / self.train_std
        input_data = X_st
        h    = X_st
        votes = []
        for i in range(self.n_layer):
            h    = tf.nn.relu(tf.matmul(h, self.weights[i]) + self.biases[i])
            H_l  = tf.concat([input_data, h], axis=1)
            batch_size = tf.shape(H_l)[0]
            H_aug = tf.concat([H_l, tf.ones([batch_size, 1], dtype=tf.float32)], axis=1)
            logits = tf.matmul(H_aug, self.betas[i])
            votes.append(tf.argmax(logits, axis=1))
        votes        = tf.stack(votes, axis=1)
        n_class      = self.betas[0].shape[1]
        votes_one_hot = tf.one_hot(tf.cast(votes, tf.int32), depth=n_class)
        summed_votes  = tf.reduce_sum(votes_one_hot, axis=1)
        return tf.argmax(summed_votes, axis=1).numpy()
 

In [ ]:

class Activation:
    @staticmethod
    def sigmoid(x): return 1 / (1 + np.e ** (-x))
    @staticmethod
    def sine(x): return np.sin(x)
    @staticmethod
    def hardlim(x): return (np.sign(x) + 1) / 2
    @staticmethod
    def tribas(x): return np.maximum(1 - np.abs(x), 0)
    @staticmethod
    def radbas(x): return np.exp(-(x**2))
    @staticmethod
    def sign(x): return np.sign(x)
    @staticmethod
    def relu(x): return np.maximum(0, x)
    @staticmethod
    def leaky_relu(x):
        x[x < 0] = x[x < 0] / 10.0
        return x


In [ ]:

def levy(n, m, beta=1.5):
    sigma = (
        math.gamma(1 + beta) *
        math.sin(math.pi * beta / 2) /
        (math.gamma((1 + beta) / 2) *
         beta * 2 ** ((beta - 1) / 2))
    ) ** (1 / beta)
    u = np.random.randn(n, m) * sigma
    v = np.random.randn(n, m)
    return u / (np.abs(v) ** (1 / beta))


In [ ]:

def true_prlu_transfer(v, n=2, alpha=0.1):
    base_term = v**2 / (1 + v**2)
    scale     = alpha / (1 + 0.05 * n)
    phi       = scale * base_term**n
    num       = v * (1 + phi)
    denom     = 1 + np.abs(v * (1 + phi))
    return np.abs(num / denom)
 
def binarize(X_continuous):
    prob        = true_prlu_transfer(X_continuous)
    rand_matrix = np.random.rand(*X_continuous.shape)
    return (rand_matrix < prob).astype(float)


In [ ]:

def wrapper_rvfl_kfold_gpu(sFeat, label, k_folds, n_class):
    kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    all_true, all_pred = [], []
    for train_idx, test_idx in kfold.split(sFeat):
        xtrain, ytrain = sFeat[train_idx], label[train_idx]
        xtest,  ytest  = sFeat[test_idx],  label[test_idx]
        model = EnsembleDeepRVFL_GPU(n_nodes=100, lam=0.01, n_layer=10)
        model.train(xtrain, ytrain, n_class)
        pred = model.predict(xtest)
        all_true.extend(ytest)
        all_pred.extend(pred)
    return np.array(all_true), np.array(all_pred)

In [ ]:

def calculate_ovr_metrics(y_true, y_pred):
    unique_labels = np.unique(y_true)
    class_metrics = {}
    for class_label in unique_labels:
        y_true_b = (y_true == class_label).astype(int)
        y_pred_b = (y_pred == class_label).astype(int)
        TP = np.sum((y_true_b == 1) & (y_pred_b == 1))
        FP = np.sum((y_true_b == 0) & (y_pred_b == 1))
        FN = np.sum((y_true_b == 1) & (y_pred_b == 0))
        TN = np.sum((y_true_b == 0) & (y_pred_b == 0))
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        accuracy    = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0.0
        class_metrics[class_label] = {
            'sensitivity': sensitivity, 'specificity': specificity,
            'accuracy': accuracy, 'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN
        }
    return class_metrics

In [ ]:

def calculate_overall_f1_ovr(y_true, y_pred):
    unique_labels   = np.unique(y_true)
    total           = len(y_true)
    class_metrics   = {}
    weighted_f1_sum = 0.0
    for class_label in unique_labels:
        y_true_b = (y_true == class_label).astype(int)
        y_pred_b = (y_pred == class_label).astype(int)
        TP = np.sum((y_true_b == 1) & (y_pred_b == 1))
        FP = np.sum((y_true_b == 0) & (y_pred_b == 1))
        FN = np.sum((y_true_b == 1) & (y_pred_b == 0))
        TN = np.sum((y_true_b == 0) & (y_pred_b == 0))
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        weight    = np.sum(y_true_b)
        class_metrics[class_label] = {
            'Precision': precision, 'Recall': recall, 'F1_Score': f1,
            'Support': weight, 'Weight': weight / total
        }
        weighted_f1_sum += f1 * weight
    overall_f1 = weighted_f1_sum / total if total > 0 else 0.0
    return overall_f1, class_metrics
 

In [ ]:

def calculate_pal_exponential_entropy(X):
    unique, counts = np.unique(X, return_counts=True)
    probabilities  = counts / len(X)
    return 1 - np.sum(np.exp(-probabilities))
 
def calculate_normalized_pal_mi(S, Y):
    H_PP_S  = calculate_pal_exponential_entropy(S)
    H_PP_Y  = calculate_pal_exponential_entropy(Y)
    joint_data   = np.column_stack([S, Y])
    joint_tuples = [tuple(row) for row in joint_data]
    unique_joints, counts = np.unique(joint_tuples, axis=0, return_counts=True)
    joint_probs  = counts / len(joint_tuples)
    H_PP_SY = 1 - np.sum(np.exp(-joint_probs))
    I_PP        = H_PP_S + H_PP_Y - H_PP_SY
    max_entropy = max(H_PP_S, H_PP_Y)
    return np.clip(I_PP / max_entropy, 0, 1) if max_entropy > 0 else 0.0
 
def calculate_renyi_entropy(X, alpha=2):
    unique, counts = np.unique(X, return_counts=True)
    probabilities  = counts / len(X)
    sum_p_alpha    = np.sum(probabilities ** alpha)
    return (1 / (1 - alpha)) * np.log(sum_p_alpha) if sum_p_alpha > 0 else 0.0
 
def calculate_normalized_renyi_mi(S, Y, alpha=2):
    H_alpha_S = calculate_renyi_entropy(S, alpha)
    H_alpha_Y = calculate_renyi_entropy(Y, alpha)
    joint_data   = np.column_stack([S, Y])
    joint_tuples = [tuple(row) for row in joint_data]
    unique_joints, counts = np.unique(joint_tuples, axis=0, return_counts=True)
    joint_probs  = counts / len(joint_tuples)
    sum_p_alpha  = np.sum(joint_probs ** alpha)
    H_alpha_SY   = (1 / (1 - alpha)) * np.log(sum_p_alpha) if sum_p_alpha > 0 else 0.0
    I_alpha      = H_alpha_S + H_alpha_Y - H_alpha_SY
    max_entropy  = max(H_alpha_S, H_alpha_Y)
    return np.clip(I_alpha / max_entropy, 0, 1) if max_entropy > 0 else 0.0
 
def calculate_gmean_ovr(y_true, y_pred):
    
    unique_labels = np.unique(y_true)
    cm            = confusion_matrix(y_true, y_pred, labels=unique_labels)
    n_classes     = len(unique_labels)
    if n_classes == 0:
        return 0.0
    gmean_per_class = []
    for i in range(n_classes):
        tp   = cm[i, i]
        fn   = cm[i, :].sum() - tp
        fp   = cm[:, i].sum() - tp
        tn   = cm.sum() - (tp + fn + fp)
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        gmean_per_class.append(np.sqrt(sens * spec) if (sens > 0 and spec > 0) else 0.0)
   
    if all(g > 0 for g in gmean_per_class):
        return float(np.prod(gmean_per_class) ** (1.0 / n_classes))
    else:
        return 0.0
 
def your_hybrid_fitness_gpu(feat, label, X, k_folds=10, n_class=None,
                             w_I=0.35, w_B=0.30, w_A=0.15, w_F=0.20, alpha=2):
    n_selected = np.sum(X == 1)
    n_total    = feat.shape[1]
    if n_selected == 0: return 100.0
    if n_selected == 1: return 50.0
    selected_features = np.where(X == 1)[0]
    sFeat = feat[:, selected_features]
    y_true, y_pred = wrapper_rvfl_kfold_gpu(sFeat, label, k_folds, n_class)
 
    I_PP_normalized    = calculate_normalized_pal_mi(y_pred, y_true)
    I_renyi_normalized = calculate_normalized_renyi_mi(y_pred, y_true, alpha=alpha)
    
    hybrid_I_normalized = (I_PP_normalized + I_renyi_normalized) / 2
 
    gmean    = calculate_gmean_ovr(y_true, y_pred)
    accuracy = np.sum(y_true == y_pred) / len(y_true)
    
    feature_ratio = n_selected / n_total
 
    if   feature_ratio < 0.05:  feature_cost = 2.0 * (0.05 - feature_ratio) ** 2
    elif feature_ratio < 0.10:  feature_cost = 0.5 * (0.10 - feature_ratio) ** 2
    elif feature_ratio <= 0.40: feature_cost = 0.01 * (feature_ratio - 0.25) ** 2
    elif feature_ratio <= 0.60: feature_cost = 0.3  * (feature_ratio - 0.40) ** 2.5
    else:                       feature_cost = 1.0  * (feature_ratio - 0.60) ** 3
 
    fitness = (w_I * (1 - hybrid_I_normalized) +
               w_B * (1 - gmean) +
               w_A * (1 - accuracy) +
               w_F * feature_cost)
 
    unique_labels = np.unique(y_true)
    cm            = confusion_matrix(y_true, y_pred, labels=unique_labels)
    class_accs    = [cm[i, i] / np.sum(cm[i, :]) if np.sum(cm[i, :]) > 0 else 0
                     for i in range(len(unique_labels))]
    min_acc = np.min(class_accs) if class_accs else 0
    if   min_acc < 0.30: fitness *= 1.50
    elif min_acc < 0.50: fitness *= 1.25
    if accuracy >= 0.90 and gmean >= 0.85 and feature_ratio <= 0.30:
        fitness *= 0.70
    return max(0.0, min(fitness, 100.0))
 

In [ ]:
import numpy as np

def compute_sigma(i, X, idx_M, Kbest):
    
    K_best_indices = idx_M[:Kbest]
    X_avg = np.mean(X[K_best_indices], axis=0)
    return np.linalg.norm(X[i] - X_avg)


def jBASO(feat, label, N, max_Iter, alpha, beta, k_folds):
    
    fun     = your_hybrid_fitness_gpu
    dim     = feat.shape[1]
    eps     = 1e-12
    n_class = len(np.unique(label))

    # ── Initialisation ───────────────────────────────────────────────────
    X    = (np.random.rand(N, dim) > 0.5).astype(float)   
    V    = np.zeros((N, dim))                               
    Vmax = 6

    fit  = np.zeros(N)
    fitG = np.inf
    Xgb  = None
    curve = np.zeros(max_Iter)

    t = 1
    while t <= max_Iter:

        
        for i in range(N):
            fit[i] = fun(feat, label, X[i, :], k_folds=k_folds, n_class=n_class)
            if fit[i] < fitG:
                fitG = fit[i]
                Xgb  = X[i, :].copy()

        
        fitB     = np.min(fit)
        fitW     = np.max(fit)
        M_unnorm = np.exp(-(fit - fitB) / (fitW - fitB + eps))
        M        = M_unnorm / np.sum(M_unnorm)

       
        Kbest = int(np.ceil(N - (N - 2) * np.sqrt(t / max_Iter)))
        idx_M = np.argsort(fit)          
        eta = alpha * ((1 - (t - 1) / max_Iter) ** 3) * np.exp(-20 * t / max_Iter)

       
        lam = beta * np.exp(-20 * t / max_Iter)

       
        A = np.zeros((N, dim))

        for i in range(N):
            sigma_i = compute_sigma(i, X, idx_M, Kbest)
            F_total = np.zeros(dim)

            for k_idx in range(Kbest):
                j = idx_M[k_idx]
                if i == j:
                    continue

                r_ij = np.linalg.norm(X[i, :] - X[j, :])

                
                g_t = 0.1 * np.sin(np.pi / 2 * t / max_Iter)
                h_min = 1.1 + g_t  
                h_max = 2.4
                
                h = r_ij / (sigma_i + eps)
                if h < h_min:
                    h = h_min
                elif h > h_max:
                    h = h_max
                else:
                    h

               
                F_mag = -eta * (2 * (h ** 13) - (h ** 7))

                
                F_vec = F_mag * (X[j, :] - X[i, :]) / (r_ij + eps)

                
                r_j     = np.random.rand()
                F_total += r_j * F_vec          

            
            G_vec = lam * (Xgb - X[i, :])

            
            A[i, :] = (F_total + G_vec) / (M[i] + eps)

        
        for i in range(N):
            for d in range(dim):

                
                V[i, d] = np.random.rand() * V[i, d] + A[i, d]
                V[i, d] = np.clip(V[i, d], -Vmax, Vmax)

                
                S = true_prlu_transfer(V[i, d])
                X[i, d] = 1 if np.random.rand() < S else 0

       
        curve[t - 1] = fitG
        print(f"  Iteration {t:3d}/{max_Iter} | Best Fitness: {fitG:.6f}")
        t += 1

    
    Sf = np.where(Xgb == 1)[0]
    Nf = len(Sf)
    if Nf == 0:
        Sf = np.array([np.random.randint(dim)])
        Nf = 1

    return feat[:, Sf], Sf, Nf, curve

In [ ]:
# ============================================================================
# FRIEDMAN RANKING FUNCTION
# ============================================================================
def friedman_ranking(data_matrix):
    """
    Calculate Friedman rankings for algorithm comparison.
    Lower rank = better performance.
    """
    n_datasets, n_algorithms = data_matrix.shape
    ranks = np.zeros((n_datasets, n_algorithms))
    
    for i in range(n_datasets):
        # Use negative values because higher accuracy = better = lower rank
        ranks[i, :] = rankdata(-data_matrix[i, :])
        
    return np.mean(ranks, axis=0)

In [ ]:
# ============================================================================
# EXPERIMENT PARAMETERS
# ============================================================================
filename = "/home/cseuser2/Downloads/ozone+level+detection/eighthr.data"
N        = 10
max_Iter = 50
a1       = 0.1
a2       = 0.5
k_folds  = 10
n_runs   = 20
 
print(f"\nDataset: {filename}")
print(f"Population size (N): {N}")
print(f"Maximum iterations:  {max_Iter}")
print(f"K-folds (inner CV):  {k_folds}")
print(f"Number of runs:      {n_runs}")
 
# ============================================================================
# LOAD DATA
# ============================================================================
print("\nLoading Ozone dataset...")
try:
    data  = np.genfromtxt(filename, delimiter=',', invalid_raise=False)
    feat  = data[:, 1:-1].astype(float)
    label = data[:, -1].astype(int)
except Exception:
    data  = np.loadtxt(filename, delimiter=',')
    feat  = data[:, 1:-1].astype(float)
    label = data[:, -1].astype(int)
 
feat  = np.nan_to_num(feat, nan=0.0)
global_labels = np.unique(label)
n_class       = len(global_labels)
feature_names = [f"Feature_{i}" for i in range(feat.shape[1])]
 
print(f"Dataset shape:      {feat.shape}")
print(f"Number of classes:  {n_class}")
print(f"Class distribution: {np.bincount(label)}")
 
# ============================================================================
# 80 / 20 STRATIFIED SPLIT  — NEW
# Feature selection (jBASO + k-fold CV) runs only on X_trainval.
# Final performance is reported on X_test (never seen during optimisation).
# ============================================================================
X_trainval, X_test, y_trainval, y_test = train_test_split(
    feat, label,
    test_size=0.20,
    random_state=42,
    stratify=label
)
 
print(f"\nValidation protocol: 80/20 stratified split")
print(f"  Train+Val size: {X_trainval.shape[0]} samples  "
      f"(class dist: {np.bincount(y_trainval)})")
print(f"  Test size:      {X_test.shape[0]} samples  "
      f"(class dist: {np.bincount(y_test)})")
 
# ============================================================================
# STORAGE
# ============================================================================
all_accuracies          = []
all_fitness             = []
all_selected_features   = []
all_num_features        = []
all_curves              = []
all_ovr_metrics         = []
all_f1_ovr              = []
all_sensitivities       = []
all_specificities       = []
all_class_sensitivities = {}
all_class_specificities = {}
all_class_precisions    = {}
all_class_recalls       = {}
all_class_f1s           = {}
all_run_times           = []   # NEW — for runtime comparison table
all_confusion_matrices = []    
print(f"\n{'='*70}")
print(f"RUNNING BASO — {n_runs} INDEPENDENT RUNS")
print(f"{'='*70}")
 
for run in range(n_runs):
    print(f"\n{'─'*70}")
    print(f"RUN {run + 1}/{n_runs}")
    print(f"{'─'*70}")
 
    np.random.seed(42 + run)
    random.seed(42 + run)
    tf.random.set_seed(42 + run)
 
    # ── Feature selection on trainval only — NEW ──────────────────────────
    t_start = time.time()
    sFeat_trainval, Sf, Nf, curve = jBASO(
        X_trainval, y_trainval, N, max_Iter, a1, a2, k_folds
    )
    run_time = time.time() - t_start
    all_run_times.append(run_time)
 
    # ── Final evaluation on held-out test set — NEW ───────────────────────
    X_trainval_sel = X_trainval[:, Sf]
    X_test_sel     = X_test[:, Sf]
 
    final_model = EnsembleDeepRVFL_GPU(n_nodes=100, lam=0.01, n_layer=10)
    final_model.train(X_trainval_sel, y_trainval, n_class)
    y_pred = final_model.predict(X_test_sel)
    y_true = y_test   # alias for clarity
 
    accuracy = float(np.sum(y_pred == y_true) / len(y_true))
 
    # ── Per-class confusion matrix printout — NEW (addresses Ozone concern) ─
    cm_run = confusion_matrix(y_true, y_pred, labels=global_labels)
    all_confusion_matrices.append(cm_run)

    print(f"\n  Test Confusion Matrix (Run {run+1}):")
    print(f"  {cm_run}")
    # FIX: minlength=n_class prevents crash when a class is absent from preds
    print(f"  Class distribution — true: {np.bincount(y_true, minlength=n_class)}  "
          f"pred: {np.bincount(y_pred.astype(int), minlength=n_class)}")
    for i, cls in enumerate(global_labels):
        row_sum  = cm_run[i, :].sum()
        recall_i = cm_run[i, i] / row_sum if row_sum > 0 else 0.0
        print(f"  Class {cls}: per-class recall = {recall_i:.4f}, "
              f"support = {row_sum}")
 
    # ── OvR Metrics ───────────────────────────────────────────────────────
    f1_ovr_val, class_metrics = calculate_overall_f1_ovr(y_true, y_pred)
    if n_class == 2:
        # BINARY FIX: Use standard confusion matrix unraveling
        # This prevents the macro-averaging illusion that forces Sens/Spec to 0.50
        cm_binary = confusion_matrix(y_true, y_pred, labels=global_labels)
        TN, FP, FN, TP = cm_binary.ravel()
        sens = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        ovr = {} # Keep empty to avoid errors in the storage loop below
    else:
        # MULTI-CLASS: Use OvR macro-average
        ovr  = calculate_ovr_metrics(y_true, y_pred)
        sens = np.mean([m['sensitivity'] for m in ovr.values()])
        spec = np.mean([m['specificity'] for m in ovr.values()])
  
    # ── Store ─────────────────────────────────────────────────────────────
    all_accuracies.append(accuracy)
    all_fitness.append(curve[-1])
    all_selected_features.append(Sf)
    all_num_features.append(Nf)
    all_curves.append(curve)
    all_ovr_metrics.append(ovr)
    all_f1_ovr.append(f1_ovr_val)
    all_sensitivities.append(sens)
    all_specificities.append(spec)
 
    for class_name, metrics in class_metrics.items():
        class_label = int(class_name)
        if class_label not in all_class_sensitivities:
            all_class_sensitivities[class_label] = []
            all_class_specificities[class_label] = []
            all_class_precisions[class_label]    = []
            all_class_recalls[class_label]       = []
            all_class_f1s[class_label]           = []
        all_class_sensitivities[class_label].append(metrics['Recall'])
        all_class_specificities[class_label].append(spec)
        all_class_precisions[class_label].append(metrics['Precision'])
        all_class_recalls[class_label].append(metrics['Recall'])
        all_class_f1s[class_label].append(metrics['F1_Score'])
 
    print(f"\n  Run {run+1} Summary (evaluated on held-out test set):")
    print(f"   Selected features: {Nf}/{feat.shape[1]}  "
          f"(ratio: {Nf/feat.shape[1]*100:.1f}%)")
    print(f"   Accuracy:          {accuracy*100:.2f}%")
    print(f"   Sensitivity (OvR): {sens:.4f}")
    print(f"   Specificity (OvR): {spec:.4f}")
    print(f"   F1-Score (OvR):    {f1_ovr_val:.4f}")
    print(f"   Final fitness:     {curve[-1]:.6f}")
    print(f"   Run time:          {run_time:.1f}s")
 
# ============================================================================
# AGGREGATE STATISTICS
# ============================================================================
mean_accuracy     = np.mean(all_accuracies);  std_accuracy     = np.std(all_accuracies)
mean_fitness      = np.mean(all_fitness);     std_fitness      = np.std(all_fitness)
mean_num_features = np.mean(all_num_features);std_num_features = np.std(all_num_features)
mean_sens         = np.mean(all_sensitivities);std_sens        = np.std(all_sensitivities)
mean_spec         = np.mean(all_specificities);std_spec        = np.std(all_specificities)
mean_f1           = np.mean(all_f1_ovr);      std_f1          = np.std(all_f1_ovr)
mean_runtime      = np.mean(all_run_times);   std_runtime      = np.std(all_run_times)
feature_ratio_pct = (mean_num_features / feat.shape[1]) * 100
 
class_stats = {}
for class_label in all_class_sensitivities:
    class_stats[class_label] = {
        'mean_sensitivity': np.mean(all_class_sensitivities[class_label]),
        'std_sensitivity':  np.std(all_class_sensitivities[class_label]),
        'mean_specificity': np.mean(all_class_specificities[class_label]),
        'std_specificity':  np.std(all_class_specificities[class_label]),
        'mean_precision':   np.mean(all_class_precisions[class_label]),
        'std_precision':    np.std(all_class_precisions[class_label]),
        'mean_recall':      np.mean(all_class_recalls[class_label]),
        'std_recall':       np.std(all_class_recalls[class_label]),
        'mean_f1':          np.mean(all_class_f1s[class_label]),
        'std_f1':           np.std(all_class_f1s[class_label])
    }
 
best_run_idx  = np.argmax(all_accuracies)
best_Sf       = all_selected_features[best_run_idx]
best_accuracy = all_accuracies[best_run_idx]
best_fitness  = all_fitness[best_run_idx]
best_curve    = all_curves[best_run_idx]
best_Nf       = all_num_features[best_run_idx]
 
print(f"\n{'='*70}")
print(f"BEST RUN RESULTS (Run {best_run_idx + 1})")
print(f"{'='*70}")
print(f" Selected: {best_Nf} features out of {feat.shape[1]}")
print(f" Reduction: {(feat.shape[1]-best_Nf)/feat.shape[1]*100:.1f}%")
print(f" Feature selection ratio: {best_Nf/feat.shape[1]*100:.1f}%")
print(f" Accuracy:  {best_accuracy*100:.2f}%  (on held-out test set)")
print(f" Fitness:   {best_fitness:.6f}")
 
print(f"\n{'='*70}")
print(f"COMPREHENSIVE STATISTICS OVER {n_runs} RUNS (held-out test set)")
print(f"{'='*70}")
print(f" Accuracy:                {mean_accuracy*100:.2f}% ± {std_accuracy*100:.2f}%")
print(f" Fitness:                 {mean_fitness:.6f} ± {std_fitness:.6f}")
print(f" Selected Features:       {mean_num_features:.2f} ± {std_num_features:.2f}")
print(f" Feature selection ratio: {feature_ratio_pct:.1f}% of original features")
print(f" Min Accuracy:            {np.min(all_accuracies)*100:.2f}%")
print(f" Max Accuracy:            {np.max(all_accuracies)*100:.2f}%")
print(f" Sensitivity (OvR):       {mean_sens:.4f} ± {std_sens:.4f}")
print(f" Specificity (OvR):       {mean_spec:.4f} ± {std_spec:.4f}")
print(f" F1-Score (OvR):          {mean_f1:.4f} ± {std_f1:.4f}")
print(f" Mean runtime per run:    {mean_runtime:.1f}s ± {std_runtime:.1f}s")
 
print(f"\n{'─'*70}")
print("PER-CLASS STATISTICS (averaged over all runs):")
print(f"{'─'*70}")
print(f"{'Class':<10} {'Sensitivity':<22} {'Specificity':<22} "
      f"{'Precision':<22} {'Recall':<22} {'F1-Score':<20}")
print(f"{'-'*118}")
for class_label, cstats in class_stats.items():
    print(f"{class_label:<10} "
          f"{cstats['mean_sensitivity']:.4f}±{cstats['std_sensitivity']:.4f}      "
          f"{cstats['mean_specificity']:.4f}±{cstats['std_specificity']:.4f}      "
          f"{cstats['mean_precision']:.4f}±{cstats['std_precision']:.4f}      "
          f"{cstats['mean_recall']:.4f}±{cstats['std_recall']:.4f}      "
          f"{cstats['mean_f1']:.4f}±{cstats['std_f1']:.4f}")
 
# Feature selection frequency
feature_selection_counts = Counter()
for sf in all_selected_features:
    feature_selection_counts.update(sf)
 
print(f"\n{'─'*70}")
print(f"FEATURE SELECTION FREQUENCY (across {n_runs} runs):")
print(f"{'─'*70}")
print(f"{'Feature Index':<15} {'Selected (times)':<20} {'Frequency (%)':<15}")
print(f"{'-'*50}")
for feature_idx in sorted(feature_selection_counts.keys()):
    count = feature_selection_counts[feature_idx]
    freq  = (count / n_runs) * 100
    print(f"{feature_idx:<15} {count:<20} {freq:.1f}%")
 
# ============================================================================
# BASELINE — trained on trainval, evaluated on test (same split as BASO)
# ============================================================================
print(f"\n{'='*70}")
print("STATISTICAL TESTS")
print(f"{'='*70}")
print("\nComputing Baseline (EnsembleDeepRVFL on ALL features, same 80/20 split)...")
 
baseline_accuracies = []
baseline_run_times  = []
 
for run in range(n_runs):
    np.random.seed(42 + run)
    tf.random.set_seed(42 + run)
    t_start    = time.time()
    m_baseline = EnsembleDeepRVFL_GPU(n_nodes=100, lam=0.01, n_layer=10)
    m_baseline.train(X_trainval, y_trainval, n_class)   # trainval only
    pred_bl    = m_baseline.predict(X_test)             # test only
    elapsed    = time.time() - t_start
    acc_bl     = float(np.sum(y_test == pred_bl) / len(y_test))
    baseline_accuracies.append(acc_bl)
    baseline_run_times.append(elapsed)
    print(f"Baseline Run {run+1}: Accuracy = {acc_bl*100:.2f}%  "
          f"time = {elapsed:.1f}s")
 
mean_baseline     = np.mean(baseline_accuracies)
mean_bl_runtime   = np.mean(baseline_run_times)
std_bl_runtime    = np.std(baseline_run_times)
print(f"Baseline Mean Accuracy: {mean_baseline*100:.2f}%")
 
# ── Wilcoxon ──────────────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print("WILCOXON SIGNED-RANK TEST (BASO vs Baseline, on held-out test set)")
print(f"{'─'*70}")
wilc_stat, wilc_p = scipy_stats.wilcoxon(all_accuracies, baseline_accuracies)
# Softened interpretation — reviewer requested conservative claims
if wilc_p < 0.05:
    sig      = "Significant"
    interp   = (f"Statistically significant difference detected (p={wilc_p:.4f}). "
                f"BASO shows promising improvement over the baseline under "
                f"this wrapper configuration.")
else:
    sig      = "Not Significant"
    interp   = (f"No statistically significant difference detected (p={wilc_p:.4f}). "
                f"BASO mean accuracy ({mean_accuracy*100:.2f}%) vs "
                f"Baseline ({mean_baseline*100:.2f}%). "
                f"Results suggest competitive but not conclusively superior performance.")
 
print(f"Statistic:              {wilc_stat:.4f}")
print(f"P-value:                {wilc_p:.6f}")
print(f"Significant (p < 0.05): {sig}")
print(f"Interpretation:         {interp}")
 
# ── Friedman Ranking ──────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print("FRIEDMAN RANKING ANALYSIS")
print(f"{'─'*70}")
 
data_matrix     = np.column_stack([all_accuracies, baseline_accuracies])
algorithm_names = ['BASO', 'Baseline']
avg_ranks       = friedman_ranking(data_matrix)
 
print(f"\nFriedman Average Ranks (Lower = Better):")
print(f"{'─'*40}")
for i, algo_name in enumerate(algorithm_names):
    print(f"  {algo_name:<15} : {avg_ranks[i]:.4f}")
 
best_algo_idx  = np.argmin(avg_ranks)
best_algo_name = algorithm_names[best_algo_idx]
print(f"\n→ Best by Friedman ranking: {best_algo_name}")
print(f"  Note: Friedman ranking reflects average relative performance across "
      f"{n_runs} runs; absolute differences should be interpreted alongside "
      f"the Wilcoxon test results above.")
 
# ── Runtime Comparison Table — NEW ────────────────────────────────────────
print(f"\n{'─'*70}")
print("RUNTIME COMPARISON")
print(f"{'─'*70}")
print(f"  BASO     : {mean_runtime:.1f}s ± {std_runtime:.1f}s per run")
print(f"  Baseline : {mean_bl_runtime:.1f}s ± {std_bl_runtime:.1f}s per run")
 
# ============================================================================
# RESULTS DIRECTORY
# ============================================================================
res_dir = "results_BASO_ozonelayer"
os.makedirs(res_dir, exist_ok=True)
os.makedirs(f"{res_dir}/convergence_curves", exist_ok=True)
 
avg_curve = np.mean(all_curves, axis=0)
std_curve = np.std(all_curves,  axis=0)
bins      = min(10, max(3, n_runs))
 
# ── Plot 1: Best Run Convergence ─────────────────────────────────────────
plt.figure(figsize=(10, 6))
plt.plot(range(1, max_Iter+1), best_curve, 'b-', linewidth=2,
         marker='o', markersize=4, markevery=max(1, max_Iter//10))
plt.xlabel('Iteration', fontsize=11); plt.ylabel('Fitness Value', fontsize=11)
plt.title(f'Best Run Convergence (Run {best_run_idx+1})', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(f'{res_dir}/01_Best_Run_Convergence.png', dpi=150,
            bbox_inches='tight', facecolor='white'); plt.close()
 
# ── Plot 2: Average Convergence ───────────────────────────────────────────
plt.figure(figsize=(10, 6))
plt.plot(range(1, max_Iter+1), avg_curve, 'r-', linewidth=2, label='Mean')
plt.fill_between(range(1, max_Iter+1), avg_curve-std_curve, avg_curve+std_curve,
                 alpha=0.3, color='red', label='Std Dev')
plt.xlabel('Iteration', fontsize=11); plt.ylabel('Fitness Value', fontsize=11)
plt.title(f'Average Convergence ({n_runs} runs)', fontsize=12, fontweight='bold')
plt.legend(fontsize=9); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(f'{res_dir}/02_Average_Convergence.png', dpi=150,
            bbox_inches='tight', facecolor='white'); plt.close()
 
# ── Plot 3: Selected Features Bar ─────────────────────────────────────────
plt.figure(figsize=(10, 6))
all_feat_idx = np.arange(feat.shape[1])
bar_colors   = ['#FF4444' if i in best_Sf else '#DDDDDD' for i in all_feat_idx]
plt.bar(all_feat_idx, np.ones(len(all_feat_idx)), color=bar_colors,
        alpha=0.8, edgecolor='black', linewidth=0.5)
plt.xlabel('Feature Index', fontsize=11); plt.ylabel('Selected', fontsize=11)
plt.title(f'Selected Features ({best_Nf}/{feat.shape[1]})', fontsize=12, fontweight='bold')
if len(all_feat_idx) > 20:
    step = max(1, len(all_feat_idx)//20)
    plt.xticks(all_feat_idx[::step], rotation=45)
else:
    plt.xticks(all_feat_idx)
plt.yticks([0, 1], ['Not Selected', 'Selected'])
plt.grid(True, alpha=0.3, axis='y'); plt.tight_layout()
plt.savefig(f'{res_dir}/03_Selected_Features.png', dpi=150,
            bbox_inches='tight', facecolor='white'); plt.close()
 
# ── Plot 4: Accuracy Distribution ────────────────────────────────────────
plt.figure(figsize=(10, 6))
plt.hist(np.array(all_accuracies)*100, bins=bins, color='#28A745',
         alpha=0.7, edgecolor='black', linewidth=1.2)
plt.axvline(mean_accuracy*100, color='red', linestyle='--', linewidth=2.5,
            label=f'Mean: {mean_accuracy*100:.2f}%')
plt.xlabel('Accuracy (%)', fontsize=11); plt.ylabel('Frequency', fontsize=11)
plt.title(f'Test-Set Accuracy Distribution ({n_runs} runs)',
          fontsize=12, fontweight='bold')
plt.legend(fontsize=9); plt.grid(True, alpha=0.3, axis='y'); plt.tight_layout()
plt.savefig(f'{res_dir}/04_Accuracy_Distribution.png', dpi=150,
            bbox_inches='tight', facecolor='white'); plt.close()
 
# ── Plot 5: Fitness Distribution ──────────────────────────────────────────
plt.figure(figsize=(10, 6))
plt.hist(all_fitness, bins=bins, color='#FD7E14', alpha=0.7,
         edgecolor='black', linewidth=1.2)
plt.axvline(mean_fitness, color='red', linestyle='--', linewidth=2.5,
            label=f'Mean: {mean_fitness:.4f}')
plt.xlabel('Fitness Value', fontsize=11); plt.ylabel('Frequency', fontsize=11)
plt.title(f'Fitness Distribution ({n_runs} runs)', fontsize=12, fontweight='bold')
plt.legend(fontsize=9); plt.grid(True, alpha=0.3, axis='y'); plt.tight_layout()
plt.savefig(f'{res_dir}/05_Fitness_Distribution.png', dpi=150,
            bbox_inches='tight', facecolor='white'); plt.close()
 
# ── Plot 6: Feature Count Distribution ───────────────────────────────────
plt.figure(figsize=(10, 6))
plt.hist(all_num_features, bins=bins, color='#6F42C1', alpha=0.7,
         edgecolor='black', linewidth=1.2)
plt.axvline(mean_num_features, color='red', linestyle='--', linewidth=2.5,
            label=f'Mean: {mean_num_features:.1f}')
plt.xlabel('Number of Features', fontsize=11); plt.ylabel('Frequency', fontsize=11)
plt.title(f'Feature Count Distribution ({n_runs} runs)',
          fontsize=12, fontweight='bold')
plt.legend(fontsize=9); plt.grid(True, alpha=0.3, axis='y'); plt.tight_layout()
plt.savefig(f'{res_dir}/06_Feature_Count_Distribution.png', dpi=150,
            bbox_inches='tight', facecolor='white'); plt.close()
 
# ── Plot 7: Run-by-Run Performance ────────────────────────────────────────
plt.figure(figsize=(10, 6))
x_runs = np.arange(1, n_runs+1)
plt.plot(x_runs, np.array(all_accuracies)*100, 'o-', linewidth=2.5,
         markersize=10, color='#007BFF', markeredgecolor='black',
         markeredgewidth=1, label='BASO Accuracy (test set)')
plt.axhline(mean_accuracy*100, color='red', linestyle='--', linewidth=2,
            label=f'Mean: {mean_accuracy*100:.2f}%')
plt.fill_between(x_runs,
                 (mean_accuracy-std_accuracy)*100,
                 (mean_accuracy+std_accuracy)*100,
                 alpha=0.2, color='red', label='±1 Std')
plt.xlabel('Run Number', fontsize=11); plt.ylabel('Accuracy (%)', fontsize=11)
plt.title('Run-by-Run Test Accuracy', fontsize=12, fontweight='bold')
plt.legend(fontsize=9); plt.grid(True, alpha=0.3); plt.xticks(x_runs)
plt.tight_layout()
plt.savefig(f'{res_dir}/07_Run_Performance.png', dpi=150,
            bbox_inches='tight', facecolor='white'); plt.close()
 
# ── Plot 8: Baseline Comparison Boxplot ───────────────────────────────────
plt.figure(figsize=(8, 8))
bp = plt.boxplot(
    [np.array(all_accuracies)*100, np.array(baseline_accuracies)*100],
    labels=['BASO', 'Baseline (all features)'],
    patch_artist=True,
    boxprops=dict(linewidth=1.5),
    whiskerprops=dict(linewidth=1.5),
    capprops=dict(linewidth=1.5),
    medianprops=dict(color='darkred', linewidth=2)
)
for patch, color in zip(bp['boxes'], ['#90EE90', '#FFB6C6']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
plt.ylabel('Accuracy (%) on held-out test set', fontsize=11)
plt.title('BASO vs Baseline — Test Set Accuracy', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y'); plt.tight_layout()
plt.savefig(f'{res_dir}/08_Baseline_Comparison.png', dpi=150,
            bbox_inches='tight', facecolor='white'); plt.close()
 
# ── Plot 9: Feature Selection Frequency ──────────────────────────────────
if len(feature_selection_counts) > 0:
    plt.figure(figsize=(14, 6))
    sorted_feats = sorted(feature_selection_counts.keys())
    freqs        = [feature_selection_counts[f] for f in sorted_feats]
    bar_cols     = ['#FF4444' if freq == n_runs
                    else '#FFA500' if freq >= n_runs/2
                    else '#90EE90' for freq in freqs]
    plt.bar(range(len(sorted_feats)), freqs, color=bar_cols,
            alpha=0.8, edgecolor='black', linewidth=1)
    plt.axhline(n_runs/2, color='blue', linestyle='--', linewidth=2,
                alpha=0.6, label=f'50% threshold ({n_runs/2:.1f})')
    plt.axhline(n_runs,   color='red',  linestyle='--', linewidth=2,
                alpha=0.6, label=f'100% ({n_runs})')
    plt.xlabel('Feature Index', fontsize=12, fontweight='bold')
    plt.ylabel('Selection Frequency', fontsize=12, fontweight='bold')
    plt.title(f'Feature Selection Frequency Across {n_runs} Runs',
              fontsize=14, fontweight='bold', pad=15)
    plt.xticks(range(len(sorted_feats)), sorted_feats, rotation=45)
    plt.legend(fontsize=10); plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f'{res_dir}/09_Feature_Selection_Frequency.png', dpi=150,
                bbox_inches='tight', facecolor='white'); plt.close()
 
# ── Plot 10: All Convergence Curves ──────────────────────────────────────
plt.figure(figsize=(14, 7))
colors_palette = plt.cm.tab10(np.linspace(0, 1, max(n_runs, 10)))
for i, crv in enumerate(all_curves):
    plt.plot(range(1, max_Iter+1), crv, alpha=0.4, linewidth=1.5,
             color=colors_palette[i % 10],
             label=f'Run {i+1}' if n_runs <= 10 else '')
plt.plot(range(1, max_Iter+1), avg_curve, 'r-', linewidth=3.5,
         label='Average', zorder=10, marker='o', markersize=6,
         markevery=max(1, max_Iter//10))
plt.xlabel('Iteration', fontsize=13, fontweight='bold')
plt.ylabel('Fitness Value', fontsize=13, fontweight='bold')
plt.title(f'Convergence Curves — All {n_runs} Runs',
          fontsize=15, fontweight='bold', pad=15)
plt.legend(fontsize=10 if n_runs <= 10 else 11, loc='best')
plt.grid(True, alpha=0.3, linestyle='--', linewidth=0.7); plt.tight_layout()
plt.savefig(f'{res_dir}/10_All_Convergence.png', dpi=150,
            bbox_inches='tight', facecolor='white'); plt.close()
 
# ── Plot 11: Per-class Radar Chart ────────────────────────────────────────
if len(class_stats) > 0:
    num_classes = len(class_stats)
    fig, axes = plt.subplots(1, min(num_classes, 2), figsize=(14, 6),
                             subplot_kw=dict(projection='polar'))
    if num_classes == 1: axes = [axes]
    categories = ['Sensitivity', 'Specificity', 'Precision', 'Recall', 'F1-Score']
    angles     = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
    angles    += angles[:1]
    for idx, (class_label, cstats) in enumerate(list(class_stats.items())[:2]):
        ax     = axes[idx]
        values = [cstats['mean_sensitivity']*100, cstats['mean_specificity']*100,
                  cstats['mean_precision']*100,   cstats['mean_recall']*100,
                  cstats['mean_f1']*100]
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2.5, color='#FF6B6B', markersize=8)
        ax.fill(angles, values, alpha=0.25, color='#FF6B6B')
        ax.set_xticks(angles[:-1]); ax.set_xticklabels(categories, fontsize=10)
        ax.set_ylim(0, 100); ax.set_yticks([20, 40, 60, 80, 100])
        ax.set_yticklabels(['20%','40%','60%','80%','100%'], fontsize=9)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.set_title(f'Class: {class_label}', fontsize=12,
                     fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f'{res_dir}/11_PerClass_Radar.png', dpi=150,
                bbox_inches='tight', facecolor='white'); plt.close()
 
# ── Plot 12: Friedman Ranking ─────────────────────────────────────────────
plt.figure(figsize=(8, 6))
y_pos  = np.arange(len(algorithm_names))
colors = ['#4CAF50' if i == best_algo_idx else '#FFC107'
          for i in range(len(algorithm_names))]
plt.barh(y_pos, avg_ranks, color=colors, edgecolor='black', linewidth=1.2)
plt.yticks(y_pos, algorithm_names)
plt.xlabel('Average Friedman Rank (Lower = Better)', fontsize=11)
plt.title('Friedman Ranking: BASO vs Baseline', fontsize=12, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x'); plt.tight_layout()
plt.savefig(f'{res_dir}/12_Friedman_Ranking.png', dpi=150,
            bbox_inches='tight', facecolor='white'); plt.close()
 
# ── Plot 13: Runtime Comparison — NEW ─────────────────────────────────────
plt.figure(figsize=(8, 5))
rt_means = [mean_runtime, mean_bl_runtime]
rt_stds  = [std_runtime,  std_bl_runtime]
plt.bar(['BASO', 'Baseline (all features)'], rt_means, yerr=rt_stds,
        color=['#4CAF50', '#FFC107'], edgecolor='black', linewidth=1.2,
        capsize=6, alpha=0.85)
plt.ylabel('Mean Runtime per Run (seconds)', fontsize=11)
plt.title('Runtime Comparison', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y'); plt.tight_layout()
plt.savefig(f'{res_dir}/13_Runtime_Comparison.png', dpi=150,
            bbox_inches='tight', facecolor='white'); plt.close()
 
# ============================================================================
# CSV OUTPUTS
# ============================================================================
iterations = np.arange(1, max_Iter+1)
 
pd.DataFrame(np.array(all_curves), columns=[f"Iter_{i}" for i in iterations]
             ).assign(Run=range(1, n_runs+1)
             ).to_csv(f'{res_dir}/Convergence_PerRun.csv', index=False)
 
pd.DataFrame({'Iteration': iterations,
              'Mean_Fitness': avg_curve,
              'Std_Fitness':  std_curve}
             ).to_csv(f'{res_dir}/Convergence_Average.csv', index=False)
 
pd.DataFrame({
    'Run':               list(range(1, n_runs+1)),
    'Accuracy_%':        [a*100 for a in all_accuracies],
    'Fitness':           all_fitness,
    'Num_Features':      all_num_features,
    'Feature_Ratio_%':   [n/feat.shape[1]*100 for n in all_num_features],
    'Sensitivity_OvR':   all_sensitivities,
    'Specificity_OvR':   all_specificities,
    'F1_OvR':            all_f1_ovr,
    'Runtime_s':         all_run_times,
    'Selected_Features': [str(sf.tolist()) for sf in all_selected_features]
}).to_csv(f'{res_dir}/All_Runs_Detailedbasoozone.csv', index=False)
 
pd.DataFrame({
    'Metric': ['Mean Accuracy (%)', 'Std Accuracy (%)',
               'Mean Fitness',      'Std Fitness',
               'Mean Num Features', 'Std Num Features',
               'Feature Ratio (%)', 'Mean Sensitivity', 'Std Sensitivity',
               'Mean Specificity',  'Std Specificity',
               'Mean F1-Score (OvR)', 'Std F1-Score (OvR)',
               'Best Accuracy (%)', 'Best Fitness',
               'Mean Runtime (s)',   'Std Runtime (s)'],
    'Value':  [mean_accuracy*100,   std_accuracy*100,
               mean_fitness,         std_fitness,
               mean_num_features,    std_num_features,
               feature_ratio_pct,    mean_sens,   std_sens,
               mean_spec,            std_spec,
               mean_f1,              std_f1,
               best_accuracy*100,    best_fitness,
               mean_runtime,         std_runtime]
}).to_csv(f'{res_dir}/Summary_Statistics.csv', index=False)
 
pd.DataFrame({
    'Test':                  ['Wilcoxon Signed-Rank'],
    'Statistic':             [wilc_stat],
    'P_value':               [wilc_p],
    'Significant_at_0.05':   [sig],
    'Interpretation':        [interp]
}).to_csv(f'{res_dir}/wilcoxon_baseline.csv', index=False)
 
pd.DataFrame({
    'Run':                list(range(1, n_runs+1)),
    'BASO_Accuracy_%':    [a*100 for a in all_accuracies],
    'Baseline_Accuracy_%':[a*100 for a in baseline_accuracies]
}).to_csv(f'{res_dir}/baseline_vs_algorithm_accuracies.csv', index=False)
 
pd.DataFrame({
    'Algorithm':      algorithm_names,
    'Avg_Rank':       avg_ranks,
    'Best_Algorithm': best_algo_name
}).to_csv(f'{res_dir}/friedman_ranking.csv', index=False)
 
# Runtime table — NEW (addresses reviewer efficiency claim)
pd.DataFrame({
    'Algorithm':      ['BASO', 'Baseline (all features)'],
    'Mean_Time_s':    [mean_runtime,    mean_bl_runtime],
    'Std_Time_s':     [std_runtime,     std_bl_runtime]
}).to_csv(f'{res_dir}/runtime_comparison.csv', index=False)
 
# ============================================================================
# FINAL SUMMARY
# ============================================================================
print(f"\n{'='*70}")
print("FINAL SUMMARY")
print(f"{'='*70}")
print(f"Dataset:            ozone")
print(f"Total samples:      {feat.shape[0]}")
print(f"Original features:  {feat.shape[1]}")
print(f"Validation:         80% train+val (10-fold CV inside optimizer) / "
      f"20% held-out test")
print(f"\nBest Run (Run {best_run_idx+1}):")
print(f"  Selected:          {best_Nf} features  "
      f"(ratio: {best_Nf/feat.shape[1]*100:.1f}%)")
print(f"  Reduction:         {(feat.shape[1]-best_Nf)/feat.shape[1]*100:.1f}%")
print(f"  Test Accuracy:     {best_accuracy*100:.2f}%")
print(f"  Fitness:           {best_fitness:.6f}")
print(f"\nStatistics over {n_runs} runs (held-out test set):")
print(f"  Accuracy:          {mean_accuracy*100:.2f}% ± {std_accuracy*100:.2f}%")
print(f"  Fitness:           {mean_fitness:.6f} ± {std_fitness:.6f}")
print(f"  Selected Features: {mean_num_features:.2f} ± {std_num_features:.2f} "
      f"({feature_ratio_pct:.1f}% of original)")
print(f"  Sensitivity (OvR): {mean_sens:.4f} ± {std_sens:.4f}")
print(f"  Specificity (OvR): {mean_spec:.4f} ± {std_spec:.4f}")
print(f"  Mean Runtime:      {mean_runtime:.1f}s ± {std_runtime:.1f}s")
print(f"\nWilcoxon vs Baseline: {sig} (p={wilc_p:.4f})")
print(f"Friedman best:        {best_algo_name}")
print(f"\n{'='*70}")
print(f"BASO FEATURE SELECTION COMPLETED SUCCESSFULLY!")
print(f"All results saved to: {res_dir}/")
print(f"{'='*70}")

# ============================================================================
# AGGREGATED CONFUSION MATRIX
# ============================================================================
print(f"\n{'='*70}")
print("GENERATING AGGREGATED CONFUSION MATRIX")
print(f"{'='*70}")

# FIX: np.stack builds a true 3D array (n_runs, n_class, n_class).
# This is safe because every run's cm_run was built with labels=global_labels,
# guaranteeing identical shape regardless of which classes appeared in y_true.
# np.mean / np.std then reduce along axis=0 correctly for both binary and
# multiclass cases — no special-casing required.
cm_stack = np.stack(all_confusion_matrices, axis=0)   # (n_runs, C, C)
avg_cm   = np.round(np.mean(cm_stack, axis=0)).astype(int)
std_cm   = np.round(np.std(cm_stack,  axis=0), 1)

print(f"Aggregated Confusion Matrix (mean over {n_runs} runs):")
print(avg_cm)
# Save mean matrix as CSV
pd.DataFrame(avg_cm,
             index=[f"True_Class_{c}" for c in global_labels],
             columns=[f"Pred_Class_{c}" for c in global_labels]
).to_csv(f'{res_dir}/Aggregated_Confusion_Matrix.csv')
# Heatmap plot — mean values, integer-formatted
plt.figure(figsize=(10, 8))
sns.heatmap(avg_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'Class {c}' for c in global_labels],
            yticklabels=[f'Class {c}' for c in global_labels])
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.title(f'Aggregated Confusion Matrix (mean over {n_runs} runs)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{res_dir}/14_Aggregated_Confusion_Matrix.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.close()


In [ ]:


import os
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats


base_dir = "/home/cseuser2/summary"


datasets = [
    "cardiotocography ",
    "isolet",
    "muskv1",
    "ozone",
    "soybean",
    "spambase",
    "statlog"
]


dataset_display_names = {
    "cardiotocography ": "Cardiotocography",
    "isolet":           "ISOLET",
    "muskv1":           "Musk V1",
    "ozone":            "Ozone Level Detection",
    "soybean":          "Soybean",
    "spambase":         "Spambase",
    "statlog":          "Statlog"
}


algo_file_keys = {
    "BASO": "baso",
    "BGWO": "bgwo",
    "BMPA": "bmpa",
    "BWOA": "bwoa"
}


ACCURACY_COL = "Accuracy_%"



def load_accuracies(folder, algo_key, dataset_name):
    """Load accuracy values from the All_Runs_Detailed CSV."""
    # Strip trailing space from dataset_name just in case
    dataset_name = dataset_name.strip()
    
    patterns = [
        f"All_Runs_Detailed_{algo_key}{dataset_name}.csv",       
        f"All_Runs_Detailed_{algo_key}_{dataset_name}.csv",     
        f"All_Runs_Detailed{algo_key}{dataset_name}.csv",        
        f"All_Runs_Detailed{algo_key.upper()}{dataset_name}.csv",
        f"All_Runs_Detailed_{algo_key.upper()}{dataset_name}.csv",
        f"All_Runs_Detailed_{algo_key.upper()}_{dataset_name}.csv",
    ]
    for fname in patterns:
        fpath = os.path.join(folder, fname)
        if os.path.exists(fpath):
            print(f"    Found: {fname}")
            df = pd.read_csv(fpath)
            if ACCURACY_COL in df.columns:
                return df[ACCURACY_COL].values / 100.0
            else:
                acc_cols = [c for c in df.columns if 'ccuracy' in c or 'acc' in c.lower()]
                if acc_cols:
                    vals = df[acc_cols[0]].values
                    return vals / 100.0 if vals.mean() > 1.0 else vals

    print(f"     {algo_key.upper()} not found in {dataset_name}. Files available:")
    for f in sorted(os.listdir(folder)):
        if f.endswith('.csv'):
            print(f"       → {f}")
    return None



def holm_bonferroni(p_values):
    
    n = len(p_values)
    indexed = sorted(enumerate(p_values), key=lambda x: x[1])
    corrected = [None] * n
    for rank, (orig_idx, p) in enumerate(indexed):
        corrected[orig_idx] = min(p * (n - rank), 1.0)
    
    sorted_corrected = [corrected[i] for i, _ in sorted(enumerate(p_values), key=lambda x: x[1])]
    for i in range(1, n):
        sorted_corrected[i] = max(sorted_corrected[i], sorted_corrected[i-1])
  
    result = [None] * n
    for rank, (orig_idx, _) in enumerate(sorted(enumerate(p_values), key=lambda x: x[1])):
        result[orig_idx] = min(sorted_corrected[rank], 1.0)
    return result


comparisons = [("BASO", "BGWO"), ("BASO", "BMPA"), ("BASO", "BWOA")]

results = []

raw_p_per_comparison = {pair: [] for pair in comparisons}
dataset_order = []

print("\n" + "="*80)
print("WILCOXON SIGNED-RANK TEST (BASO vs competitors)")
print("="*80)

for dataset in datasets:
    folder = os.path.join(base_dir, dataset)
    if not os.path.isdir(folder):
        print(f"\n  Folder not found: {folder} — skipping")
        continue

    display_name = dataset_display_names[dataset]
    dataset_order.append(dataset)
    row = {"Dataset": display_name}

    for algo1, algo2 in comparisons:
        acc1 = load_accuracies(folder, algo_file_keys[algo1], dataset)
        acc2 = load_accuracies(folder, algo_file_keys[algo2], dataset)

        if acc1 is None:
            print(f"   {algo1} file not found in {dataset}")
            row[f"{algo1}_vs_{algo2}_raw"]  = np.nan
            row[f"{algo1}_vs_{algo2}_corr"] = np.nan
            raw_p_per_comparison[(algo1, algo2)].append(np.nan)
            continue
        if acc2 is None:
            print(f"   {algo2} file not found in {dataset}")
            row[f"{algo1}_vs_{algo2}_raw"]  = np.nan
            row[f"{algo1}_vs_{algo2}_corr"] = np.nan
            raw_p_per_comparison[(algo1, algo2)].append(np.nan)
            continue

        # Wilcoxon signed-rank test
        try:
            stat, p_raw = scipy_stats.wilcoxon(acc1, acc2)
        except ValueError as e:
            # If all differences are zero, Wilcoxon cannot be computed
            print(f"  ⚠️  {algo1} vs {algo2} on {dataset}: {e}")
            p_raw = 1.0
            stat  = 0.0

        row[f"{algo1}_vs_{algo2}_raw"]  = round(p_raw, 3)
        row[f"{algo1}_vs_{algo2}_stat"] = round(stat, 4)
        raw_p_per_comparison[(algo1, algo2)].append(p_raw)

    results.append(row)


for algo1, algo2 in comparisons:
    raw_ps = [r.get(f"{algo1}_vs_{algo2}_raw", np.nan) for r in results]
    valid_mask = [not np.isnan(p) for p in raw_ps]
    valid_ps   = [p for p, v in zip(raw_ps, valid_mask) if v]

    if valid_ps:
        corrected = holm_bonferroni(valid_ps)
        corr_iter = iter(corrected)
        for r, valid in zip(results, valid_mask):
            r[f"{algo1}_vs_{algo2}_corr"] = round(next(corr_iter), 3) if valid else np.nan
    else:
        for r in results:
            r[f"{algo1}_vs_{algo2}_corr"] = np.nan



print("\nTable 14: Wilcoxon signed-rank test results (Holm-corrected p-values)")
print("-"*100)

header = f"{'Dataset':<30}"
for algo1, algo2 in comparisons:
    header += f"  {algo1} vs {algo2} (p_raw, p_H)"
print(header)
print("-"*100)

for r in results:
    line = f"{r['Dataset']:<30}"
    for algo1, algo2 in comparisons:
        p_raw  = r.get(f"{algo1}_vs_{algo2}_raw",  np.nan)
        p_corr = r.get(f"{algo1}_vs_{algo2}_corr", np.nan)
        sig_raw  = "*" if (not np.isnan(p_raw)  and p_raw  < 0.05) else " "
        sig_corr = "*" if (not np.isnan(p_corr) and p_corr < 0.05) else " "
        line += f"  {p_raw:.3f}{sig_raw}, {p_corr:.3f}{sig_corr}    "
    print(line)

print("-"*100)
print("* = significant at α=0.05")



df_out = pd.DataFrame(results)

col_order = ["Dataset"]
for algo1, algo2 in comparisons:
    col_order += [f"{algo1}_vs_{algo2}_raw", f"{algo1}_vs_{algo2}_corr"]
df_out = df_out[[c for c in col_order if c in df_out.columns]]

# Rename columns to match paper format
rename_map = {}
for algo1, algo2 in comparisons:
    rename_map[f"{algo1}_vs_{algo2}_raw"]  = f"{algo1} vs {algo2} p_raw"
    rename_map[f"{algo1}_vs_{algo2}_corr"] = f"{algo1} vs {algo2} p_H (Holm)"
df_out.rename(columns=rename_map, inplace=True)

output_csv = os.path.join(base_dir, "Table14_Wilcoxon_Results.csv")
df_out.to_csv(output_csv, index=False)
print(f"\n✅ Results saved to: {output_csv}")


print("\n" + "="*80)
print("SUMMARY OF SIGNIFICANT IMPROVEMENTS (Holm-corrected, α=0.05)")
print("="*80)

total_comparisons = 0
total_significant = 0

for algo1, algo2 in comparisons:
    col = f"{algo1} vs {algo2} p_H (Holm)"
    if col in df_out.columns:
        sig_count = (df_out[col] < 0.05).sum()
        total     = df_out[col].notna().sum()
        total_comparisons += total
        total_significant += sig_count
        print(f"  {algo1} vs {algo2}: {sig_count}/{total} significant")
        for _, row in df_out.iterrows():
            p = row[col]
            if not np.isnan(p) and p < 0.05:
                print(f"     {row['Dataset']} (p_H={p:.3f})")

print(f"\nTotal: {total_significant}/{total_comparisons} significant improvements")
print("="*80)

In [ ]:


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from scipy.stats import rankdata


base_dir = "/home/cseuser2/summary"

datasets = [
    "cardiotocography ",  
    "isolet",
    "muskv1",
    "ozone",
    "soybean",
    "spambase",
    "statlog"
]

dataset_display_names = {
    "cardiotocography ": "Cardiotocography",
    "isolet":            "ISOLET",
    "muskv1":            "Musk V1",
    "ozone":             "Ozone Level Detection",
    "soybean":           "Soybean",
    "spambase":          "Spambase",
    "statlog":           "Statlog"
}

algorithms = ["BASO", "BGWO", "BMPA", "BWOA"]

algo_file_keys = {
    "BASO": "baso",
    "BGWO": "bgwo",
    "BMPA": "bmpa",
    "BWOA": "bwoa"
}

ACCURACY_COL = "Accuracy_%"



def load_accuracies(folder, algo_key, dataset_name):
    dataset_name = dataset_name.strip()
    patterns = [
        f"All_Runs_Detailed_{algo_key}{dataset_name}.csv",
        f"All_Runs_Detailed_{algo_key}_{dataset_name}.csv",
        f"All_Runs_Detailed{algo_key}{dataset_name}.csv",
        f"All_Runs_Detailed{algo_key.upper()}{dataset_name}.csv",
        f"All_Runs_Detailed_{algo_key.upper()}{dataset_name}.csv",
        f"All_Runs_Detailed_{algo_key.upper()}_{dataset_name}.csv",
    ]
    for fname in patterns:
        fpath = os.path.join(folder, fname)
        if os.path.exists(fpath):
            df = pd.read_csv(fpath)
            if ACCURACY_COL in df.columns:
                return df[ACCURACY_COL].values / 100.0
            else:
                acc_cols = [c for c in df.columns if 'ccuracy' in c or 'acc' in c.lower()]
                if acc_cols:
                    vals = df[acc_cols[0]].values
                    return vals / 100.0 if vals.mean() > 1.0 else vals

    print(f"     {algo_key.upper()} not found in {dataset_name.strip()}")
    return None


print("\n" + "="*70)
print("LOADING DATA")
print("="*70)

data_matrix   = []
valid_datasets = []
display_names  = []

for dataset in datasets:
    folder = os.path.join(base_dir, dataset)
    if not os.path.isdir(folder):
        print(f"  Folder not found: {folder} — skipping")
        continue

    row = []
    all_found = True
    for algo in algorithms:
        accs = load_accuracies(folder, algo_file_keys[algo], dataset)
        if accs is None:
            all_found = False
            break
        row.append(np.mean(accs) * 100)  # mean accuracy in %

    if all_found:
        data_matrix.append(row)
        valid_datasets.append(dataset)
        display_names.append(dataset_display_names.get(dataset, dataset.strip()))
        print(f"{dataset_display_names.get(dataset, dataset.strip())}: "
              f"{[f'{v:.2f}%' for v in row]}")
    else:
        print(f" Skipping {dataset.strip()} — not all algorithm files found")

data_matrix = np.array(data_matrix)  # shape: (n_datasets, n_algorithms)
n_datasets, n_algorithms = data_matrix.shape

print(f"\n Loaded {n_datasets} datasets × {n_algorithms} algorithms")


print("\n" + "="*70)
print("FRIEDMAN TEST (Table 15)")
print("="*70)

algo_cols = [data_matrix[:, i] for i in range(n_algorithms)]
friedman_stat, friedman_p = scipy_stats.friedmanchisquare(*algo_cols)

print(f"\nTable 15: Friedman Test Results")
print(f"{'─'*45}")
print(f"  Statistic (χ²):        {friedman_stat:.2f}")
print(f"  Degrees of freedom:    {n_algorithms - 1}")
print(f"  p-value:               {friedman_p:.4e}")
print(f"  Significance level:    α = 0.05")
print(f"  Datasets evaluated:    {n_datasets} ({', '.join(display_names)})")
print(f"  Algorithms compared:   {n_algorithms} ({', '.join(algorithms)})")

if friedman_p < 0.05:
    conclusion = "Significant differences exist among algorithms"
    print(f"  Conclusion:            {conclusion}")
else:
    conclusion = "No significant differences detected among algorithms"
    print(f"  Conclusion:             {conclusion}")

print(f"\nInterpretation:")
print(f"  χ²={friedman_stat:.2f}, p={friedman_p:.4e} — "
      f"{'Reject' if friedman_p < 0.05 else 'Fail to reject'} null hypothesis at α=0.05.")
if friedman_p < 0.05:
    print(f"  At least one algorithm performs significantly differently from the others.")

print("\n" + "="*70)
print("FRIEDMAN AVERAGE RANKING (Table 16)")
print("="*70)


ranks_matrix = np.zeros_like(data_matrix)
for i in range(n_datasets):
    
    ranks_matrix[i, :] = rankdata(-data_matrix[i, :])

avg_ranks = np.mean(ranks_matrix, axis=0)

print(f"\nTable 16: Friedman Average Ranks")
print(f"{'─'*40}")
print(f"{'Algorithm':<15} {'Average Rank':<15} {'Position'}")
print(f"{'─'*40}")

sorted_idx = np.argsort(avg_ranks)
for pos, idx in enumerate(sorted_idx):
    marker = " ← BEST" if pos == 0 else ""
    print(f"  {algorithms[idx]:<13} {avg_ranks[idx]:.2f}{marker}")

best_algo = algorithms[np.argmin(avg_ranks)]
print(f"\n→ Best algorithm by Friedman ranking: {best_algo}")

print(f"\nPer-dataset ranks (1=best, {n_algorithms}=worst):")
print(f"{'Dataset':<25} " + "  ".join(f"{a:>6}" for a in algorithms))
print(f"{'─'*60}")
for i, dname in enumerate(display_names):
    rank_str = "  ".join(f"{ranks_matrix[i, j]:>6.0f}" for j in range(n_algorithms))
    print(f"{dname:<25} {rank_str}")
print(f"{'─'*60}")
avg_str = "  ".join(f"{avg_ranks[j]:>6.2f}" for j in range(n_algorithms))
print(f"{'Average Rank':<25} {avg_str}")




pd.DataFrame({
    'Statistic':            ['Chi-square (χ²)', 'Degrees of freedom', 'p-value',
                             'Significance level', 'Datasets evaluated',
                             'Algorithms compared', 'Conclusion'],
    'Value':                [round(friedman_stat, 2), n_algorithms - 1,
                             f"{friedman_p:.4e}", '0.05',
                             f"{n_datasets} ({', '.join(display_names)})",
                             f"{n_algorithms} ({', '.join(algorithms)})",
                             conclusion]
}).to_csv(os.path.join(base_dir, "Table15_Friedman_Test.csv"), index=False)

ranks_df = pd.DataFrame({
    'Algorithm':    algorithms,
    'Average_Rank': [round(r, 2) for r in avg_ranks]
}).sort_values('Average_Rank')
ranks_df.to_csv(os.path.join(base_dir, "Table16_Friedman_Rankings.csv"), index=False)

print(f"\n Table15_Friedman_Test.csv saved")
print(f" Table16_Friedman_Rankings.csv saved")

algorithms_plot = ["BGWO", "BASO", "BMPA", "BWOA"]
avg_ranks_plot  = [avg_ranks[algorithms.index(a)] for a in algorithms_plot]

max_rank = n_algorithms  # = 4

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))


plot_values = (max_rank + 1) - np.array(avg_ranks_plot)
plot_values_closed = list(plot_values) + [plot_values[0]]

angles = np.linspace(0, 2 * np.pi, n_algorithms, endpoint=False).tolist()
angles += angles[:1]

ax.plot(angles, plot_values_closed, 'o-', linewidth=2.5, color='#FF6B6B', markersize=8)
ax.fill(angles, plot_values_closed, alpha=0.25, color='#FF6B6B')


ax.set_xticks(angles[:-1])
ax.set_xticklabels(
    algorithms_plot,
    fontsize=14,
    fontweight='bold',
    labelpad=20        
)


ax.set_ylim(0, max_rank + 0.5)
ax.set_yticks(range(1, max_rank + 1))
ax.set_yticklabels(
    [str(max_rank - i + 1) for i in range(1, max_rank + 1)],
    fontsize=9
)


for angle, rank in zip(angles[:-1], avg_ranks_plot):
    ax.annotate(
        f'{rank:.2f}',
        xy=(angle, (max_rank + 1) - rank + 0.15),
        fontsize=11,
        fontweight='bold',
        ha='center',
        va='bottom',
        color='darkred'
    )

ax.set_title(
    'Friedman Average Ranks\n(Lower rank = Better)',
    fontsize=14,
    fontweight='bold',
    pad=20
)
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
radar_path = os.path.join(base_dir, "Figure9c_Friedman_Radar.png")
plt.savefig(radar_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ Radar chart saved to: {radar_path}")

print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
print(f"\nFriedman Test:  χ²={friedman_stat:.2f},  p={friedman_p:.4e},  "
      f"{' Significant' if friedman_p < 0.05 else ' Not Significant'}")
print(f"\nFriedman Average Rankings:")
for idx in sorted_idx:
    print(f"  {algorithms[idx]}: {avg_ranks[idx]:.2f}")
print(f"\nBest algorithm: {best_algo} (rank={avg_ranks[np.argmin(avg_ranks)]:.2f})")
print(f"\nAll results saved to: {base_dir}/")
print("="*70)